# Домашнее задание: AWB и CST

В этом домашнем задании вы реализуете методы задач баланса белого и преобразования цветовых координат, а также численно сравните их качество на предложенном датасете.

Основную часть методов надо будет имплементировать в *.py файлах, в этом .ipynb файле вам предстоит провести анализ результатов. 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import preprocess
import awb
import cst
from metrics import ColorCorrectionMetrics

## 1. Подготовка данных
 
Вам предлагается датасет из сотни гиперспектральных изображений (Hyperspectral Image, HSI, ГСИ).

В отличие от стандартного формата RGB, данные HSI содержат информацию о спектре каждой точки сцены. Здесь он предобработан, приведен к интервалу 400-700 нм с шагом дискретизации 10 нм. 

Для каждого ГСИ есть бинарная маска цветовой мишени и белого патча с нее.

Задача:
- Ознакомиться с форматом данным 
- Написать функцию, которая усредняет RGB/HSI изображение по маске белого патча 
- Реализовать линейную модель формирования изображения $ c = \int S(\lambda) I(\lambda) r(\lambda)\, d\lambda $
- Написать переход из XYZ в sRGB 
- Сравнить визуально изображения в XYZ и sRGB

Ответить на вопросы:
- Почему надо переходить в пространство sRGB, почему изображения в XYZ, зачастую, зеленоваты? 

In [ ]:
# Реализуйте все функции в preprocess.py

# Здесь покажите визуальное отличие XYZ/sRGB

# В отдельной ячейке дайте ответ на вопрос

## 2. Методы AWB

Баланс белого (AWB) нужен для:
1) Симуляции механизмов цветовой константности человека
2) Необходимый шаг для перехода в стандартные цветовые пространства (например, sRGB)

Этот этап состоит из двух шагов:
1) Определение точки белого
2) Коррекция изображения с помощью точки белого 

*В этом задании делаем коррекцию через Коррекцию Фон-Криса (поканальное деление на точку белого)*

Задача:
- Реализовать методы `GrayWorld`, `MaxRGB`, `GrayEdge` 
- Реализовать метрики `angle` и `reproduction error`
- Сравнить численно точку белого полученную методами выше и полученную с белого патча 
- Построить распределение ошибок на всем датасете, средний, квантиль, максимум. Посмотреть на изображения и ошибки на них. Понять, в каких случаях алгоритмы плохо отрабатывают.

Ответить на вопросы: 
- Какие предположения у методов `GrayWorld`, `MaxRGB`, `GrayEdge`?
- В каких сценариях сцены эти методы плохо работают?
- Почему стоит нормировать полученную точку белого на значение в зеленом канале?
- Может ли для сцены быть несколько точек белого?

In [ ]:
# Реализуйте все функции в awb.py, metrics.py

# Для каждого изображения посчитайте метрики deltaE, angle, reproduction_error
# Отобразите примеры изображений для каждого метода с самой большой ошибкой. 
# Укажите ваше предположение почему методы плохо отработали.

# В отдельной ячейке дайте ответы на вопросы

## 3. Методы CST
Преобразование цветовых координат решается:
1) переход в пространство стандартного наблюдателя  [нужно для правильного отображения на экран, например]
2) переход между RAW разных телефонов [например, для реверса пайплайна разных телефонов]

Задача: 
- Реализовать LCC/PCC/RPCC методы до четвертой степени.
- Разделить датасет на train-test выборку в 80%-20% соотношении
- Обучиться на трейне, на тесте посчитать метрики angle, deltaE. Построить гистограммы ошибок.
- В координатах цветности отобразить до-после CST положение точек и их истинное положение 

Ответить на вопросы:
- Что такое условие Максвелла-Лютера-Айвса? За счет чего оно нарушается? 
- Метод PCC не инвариантен к изменению мощности излучения (при шкалировании RGB в k-раз, компоненты PCC-вектора могут увеличиться в k^n раз). Как думаете, на сколько это критично? Можно ли найти примеры в предложенном датасете где метод по этим причинам плохо справляется?
- На практике у методов PCC/RPCC порядок не превышает четвертого. Как думаете, с чем это связано?

In [ ]:
# Реализуйте все функции в cst.py

# Разделите датасет на трейн-тест в соотношении 80%-20%. 
# Мы будем учить отображение из Canon600d в XYZ.

# Для каждого изображения посчитайте метрики deltaE, angle
# Отобразите примеры изображений для каждого метода с самой большой ошибкой. 
# Укажите ваше предположение почему методы плохо отработали.

# Перейдите в координаты цветности (x = X/(X+Y+Z), y = Y /(X+Y+Z))
# Отобразить красным цветом точки в пространстве Canon600D
# Отобразить синим цветом точки переведенные из Canon600D -> XYZ
# Отобразить зеленым цветом истинные точки XYZ (истинные точки получаются рендером ГСИ с нужными чувствительностями.) 

# В отдельной ячейке дайте ответы на вопросы

np.random.seed(42)
from sklearn.model_selection import train_test_split

## 4. Коррекция Фон-Криса
В задаче автоматического баланса белого наша цель — минимизировать влияние источника освещения I(λ) на финальное изображение. В идеальном случае мы хотим перейти от наблюдаемого сигнала c:
$$
c = \int S(\lambda) I(\lambda) r(\lambda)\, d\lambda
$$

к сигналу при эквипотенциальному источнику I(λ)=1:

$$
c_{ideal} = \int S(\lambda) r(\lambda)\, d\lambda.
$$

Поскольку в реальных камерах мы не имеем доступа к полному спектральному кубу, корректное «деление» на функцию I(λ) под интегралом невозможно. Коррекция Фон Криса предполагает, что отношение интегралов может заменить интеграл отношения:


$$
c_{von-kries} = \frac{c}{c_{wp}} = \frac{\int S(\lambda) I(\lambda) r(\lambda)\, d\lambda}{\int S(\lambda) I(\lambda)\, d\lambda}.
$$

В данном задании мы проверим, насколько отношение интегралов приближает интеграл отношения

Задача:
- Получить два варианта коррекции:
  1) Выполнить деление в спектральном пространстве (HSI разделить на спектр белого патча), после чего переведите результат в цветовое пространство XYZ.
  2) Переведите ГСИ в XYZ, затем выполните деление на координаты белого патча в этом же пространстве.
- Визуально сравнить изображения полученные через 1) и 2)
- Посчитать `angle` и `deltaE2000` между полученными изображениями

Ответить на вопросы:
- Существуют ли способы отличие от Фон-Криса для коррекции?
- Для каких спектров источника коррекция Фон-Криса будет работать хуже всего?

In [ ]:
# Получить два варианта коррекции, посчитать ошибку между этими вариантами.

# Для каждого изображения взять средний спектр с белого патча. 
# Посмотреть как форма спектра влияет на значение ошибки.
